# Maestro – Approche B : YOLOv8 + HaGRID (Google Colab)

Entraîne le détecteur de gestes YOLOv8 sur **HaGRID** (scènes réelles, plusieurs mains, objets autour)
complété par **vos images** (`collect_data.py`, seule source du geste « index à gauche »),
puis compare les approches sur les mêmes images.

**Avant de commencer :** *Exécution > Modifier le type d'exécution > GPU (T4)*.

## 1. Projet Maestro
Clone depuis GitHub (poussez vos dernières modifications avant), ou téléversez le dossier en zip.

In [ ]:
!git clone -q https://github.com/aySouhir-Ayari/maestro-gesture-control.git
%cd maestro-gesture-control
!pip install -q ultralytics mediapipe kagglehub

## 2. Vos données (optionnel mais conseillé)
Sur le PC, compressez `data/landmarks.csv` + `data/images/` en **maestro_data.zip**, puis téléversez-le ici.

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()            # choisissez maestro_data.zip (ou annulez pour HaGRID seul)
for name in up:
    zipfile.ZipFile(name).extractall('.')
print('landmarks.csv présent :', os.path.exists('data/landmarks.csv'))

## 3. Télécharger HaGRID (échantillon 30k, 384p, ~2 Go)

In [ ]:
import kagglehub
HAGRID = kagglehub.dataset_download('innominate817/hagrid-sample-30k-384p')
print(HAGRID)
!find "$HAGRID" -maxdepth 2 -type d | head

## 4. Préparer le dataset YOLO
`--per-class` : nombre d'images gardées par geste HaGRID (800 ≈ 12 000 images au total).

In [ ]:
!python scripts/prepare_hagrid.py --hagrid-ann "$HAGRID" --hagrid-img "$HAGRID" --per-class 800

## 5. Entraîner YOLOv8n (~30 à 45 min sur T4)

In [ ]:
!python scripts/train_yolo.py --epochs 50 --batch 64 --device 0

In [ ]:
from IPython.display import Image, display
display(Image('runs/maestro_val/confusion_matrix_normalized.png', width=650))
display(Image('runs/maestro/results.png', width=900))

## 6. Évaluation comparative sur les images de validation (scènes réelles HaGRID + vos images)
A1 MediaPipe + règles · A2 MediaPipe + ML (si entraîné) · B YOLOv8

In [ ]:
!python scripts/train_classifier.py || echo "(pas de landmarks.csv : A2 ignoré)"
!python scripts/evaluate.py --source yolo --limit 1500
from IPython.display import Markdown
display(Markdown(open('reports/evaluation_yolo.md', encoding='utf-8').read()))

## 7. Récupérer le modèle et les rapports
Copiez `maestro_yolo.pt` dans le dossier `models/` du projet sur le PC, puis `python main.py --classifier yolo`.

In [ ]:
!zip -qr maestro_resultats.zip models/maestro_yolo.pt reports runs/maestro_val
files.download('maestro_resultats.zip')